# Week 8 Exercise - RAG-Enhanced Price Estimation Agent

A price estimation agent that uses **Retrieval-Augmented Generation (RAG)** to find similar
products before making predictions. Instead of guessing blind, the agent first searches a
vector database of known products and their prices, then uses those as context.

**Architecture:**
1. **Vector Store** — Embed training products with sentence-transformers, store in ChromaDB
2. **RAG Agent** — Retrieve similar products, include them as context in the prompt
3. **Baseline Agent** — Same model but without RAG context
4. **Evaluation** — Head-to-head comparison to measure RAG's impact

Key Week 8 concepts: Agent architecture, RAG, vector databases, tool use.

In [ ]:
!pip install -q chromadb sentence-transformers==3.2.1 litellm datasets gradio python-dotenv huggingface_hub

In [ ]:
import os
import re
import random
import logging
import numpy as np
from tqdm import tqdm
from dotenv import load_dotenv
from huggingface_hub import login
from litellm import completion
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import chromadb

load_dotenv(override=True)
login(os.environ['HF_TOKEN'], add_to_git_credential=True)

logging.basicConfig(level=logging.INFO, format='%(message)s')

In [ ]:
ds = load_dataset("ed-donner/items_lite")
train = list(ds["train"])
test = list(ds["test"])
print(f"Train: {len(train)}, Test: {len(test)}")

## Build the Vector Store

Embed training product summaries and store them in ChromaDB so our agent can retrieve similar products at inference time.

In [ ]:
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Use a subset for speed — 2000 products is plenty for good retrieval
STORE_SIZE = 2000
store_items = random.sample(train, STORE_SIZE)

summaries = [item["summary"] for item in store_items]
prices = [item["price"] for item in store_items]

print(f"Encoding {STORE_SIZE} products...")
embeddings = encoder.encode(summaries, show_progress_bar=True)

# Store in ChromaDB
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="products")

collection.add(
    ids=[str(i) for i in range(STORE_SIZE)],
    embeddings=embeddings.tolist(),
    documents=summaries,
    metadatas=[{"price": p} for p in prices],
)
print(f"Vector store ready with {collection.count()} products")

## Define the Agents

Both agents inherit from a simple base class with colored logging (following the Week 8 Agent pattern).

In [ ]:
MODEL = "openrouter/openai/gpt-4o-mini"

# Agent colors for logging
BLUE = "\033[34m"
GREEN = "\033[32m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"


def log(agent_name, color, message):
    print(f"{color}[{agent_name}] {message}{RESET}")


def extract_price(text):
    text = str(text).replace(",", "").replace("$", "")
    match = re.search(r"\d+\.?\d*", text)
    return float(match.group()) if match else 0.0

In [ ]:
def find_similar_products(description, n_results=5):
    """RAG tool: retrieve similar products from the vector store."""
    query_embedding = encoder.encode([description])
    results = collection.query(
        query_embeddings=query_embedding.tolist(), n_results=n_results
    )
    docs = results["documents"][0]
    prices = [m["price"] for m in results["metadatas"][0]]
    return docs, prices


def rag_agent_predict(description, verbose=False):
    """RAG Agent: find similar products, then estimate price with context."""
    log("RAG Agent", BLUE, "Searching vector store for similar products...")
    similar_docs, similar_prices = find_similar_products(description)

    # Build context from similar products
    context = "Here are similar products and their known prices:\n\n"
    for doc, price in zip(similar_docs, similar_prices):
        context += f"Product: {doc[:150]}...\nPrice: ${price:.2f}\n\n"

    if verbose:
        log("RAG Agent", BLUE, f"Found {len(similar_docs)} similar products (avg ${np.mean(similar_prices):.2f})")

    message = (
        f"Estimate the price of this product. Use the similar products below as reference.\n\n"
        f"Product to estimate:\n{description}\n\n"
        f"{context}\n"
        f"Respond with ONLY a dollar amount."
    )

    response = completion(
        model=MODEL,
        messages=[{"role": "user", "content": message}],
        temperature=0.0,
    )
    price = extract_price(response.choices[0].message.content)
    log("RAG Agent", BLUE, f"Predicting ${price:.2f}")
    return price, similar_prices


def baseline_agent_predict(description, verbose=False):
    """Baseline Agent: estimate price without any context."""
    log("Baseline Agent", GREEN, "Estimating without context...")
    message = (
        f"Estimate the price of this product. Respond with ONLY a dollar amount.\n\n{description}"
    )
    response = completion(
        model=MODEL,
        messages=[{"role": "user", "content": message}],
        temperature=0.0,
    )
    price = extract_price(response.choices[0].message.content)
    log("Baseline Agent", GREEN, f"Predicting ${price:.2f}")
    return price

In [ ]:
# Test on a single item
item = test[0]
print(item["summary"][:300])
print(f"\nActual price: ${item['price']:.2f}\n")

baseline_price = baseline_agent_predict(item["summary"], verbose=True)
rag_price, similar_prices = rag_agent_predict(item["summary"], verbose=True)

print(f"\nBaseline error: ${abs(baseline_price - item['price']):.2f}")
print(f"RAG error:      ${abs(rag_price - item['price']):.2f}")

## Head-to-Head Evaluation

Does RAG context actually improve price estimation over the baseline?

In [ ]:
SIZE = 20
sample = random.sample(test, SIZE)

baseline_errors, rag_errors = [], []

for i, item in enumerate(tqdm(sample)):
    truth = item["price"]

    b_price = baseline_agent_predict(item["summary"])
    b_err = abs(b_price - truth)
    baseline_errors.append(b_err)

    r_price, _ = rag_agent_predict(item["summary"])
    r_err = abs(r_price - truth)
    rag_errors.append(r_err)

    color = GREEN if r_err < b_err else (YELLOW if r_err == b_err else RED)
    print(f"{color}#{i+1} Truth=${truth:.0f} | Baseline=${b_price:.0f} (err=${b_err:.0f}) | RAG=${r_price:.0f} (err=${r_err:.0f}){RESET}")

print(f"\n{'='*50}")
print(f"Baseline MAE: ${np.mean(baseline_errors):.2f}")
print(f"RAG MAE:      ${np.mean(rag_errors):.2f}")
improvement = np.mean(baseline_errors) - np.mean(rag_errors)
wins = sum(1 for b, r in zip(baseline_errors, rag_errors) if r < b)
print(f"RAG improvement: ${improvement:.2f} | RAG wins: {wins}/{SIZE}")

## Gradio UI

In [ ]:
import gradio as gr


def predict_and_explain(description):
    baseline = baseline_agent_predict(description)
    rag_price, similar_prices = rag_agent_predict(description)

    similar_docs, _ = find_similar_products(description)
    context_info = "Similar products found:\n\n"
    for doc, price in zip(similar_docs, similar_prices):
        context_info += f"  ${price:.2f} — {doc[:80]}...\n"
    context_info += f"\nAvg similar price: ${np.mean(similar_prices):.2f}"

    return f"${baseline:,.2f}", f"${rag_price:,.2f}", context_info


def load_random_item():
    item = random.choice(test)
    return item["summary"], f"${item['price']:.2f}"


with gr.Blocks(title="RAG Price Estimator") as demo:
    gr.Markdown("# RAG-Enhanced Price Estimation Agent")
    gr.Markdown(
        "Compare a baseline agent (no context) vs a RAG agent that retrieves "
        "similar products from a vector store before estimating."
    )

    description = gr.Textbox(lines=5, label="Product Description")

    with gr.Row():
        predict_btn = gr.Button("Estimate Price", variant="primary")
        random_btn = gr.Button("Random Test Item")

    actual_price = gr.Textbox(label="Actual Price (test items)", interactive=False)

    with gr.Row():
        baseline_out = gr.Textbox(label="Baseline (no context)")
        rag_out = gr.Textbox(label="RAG Agent (with context)")

    context_out = gr.Textbox(label="Retrieved Similar Products", lines=6)

    predict_btn.click(
        fn=predict_and_explain, inputs=description,
        outputs=[baseline_out, rag_out, context_out],
    )
    random_btn.click(fn=load_random_item, outputs=[description, actual_price])

demo.launch()

## Conclusion

By adding a RAG step — retrieving similar products from a vector store before calling the LLM — the agent gets pricing context that a standalone model doesn't have. This mirrors the FrontierAgent pattern from Week 8, where similar product lookup via ChromaDB provides grounding for price estimates.